In [3]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.neural_network import MLPRegressor

In [4]:
FILE = 'games-players-processed/games_training_set_6.csv'

### Leitura

In [5]:
df = pd.read_csv(FILE)

C:\Users\duilio\AppData\Local\Temp\ipykernel_7896\1324869264.py:1: DtypeWarning: Columns (0: seasonYear_home_p1_home_x, 1: seasonYear_home_p2_home_x, 2: seasonYear_home_p3_home_x, 3: seasonYear_away_p1_home_x, 4: seasonYear_away_p2_home_x, 5: seasonYear_away_p3_home_x, 6: seasonYear_home_p1_away_x, 7: seasonYear_home_p2_away_x, 8: seasonYear_home_p3_away_x, 9: seasonYear_away_p1_away_x, 10: seasonYear_away_p2_away_x, 11: seasonYear_away_p3_away_x, 12: seasonYear_y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILE)


In [6]:
df = df.select_dtypes(exclude=['category', 'object'])
df = df.fillna(0)
df = df.T.drop_duplicates().T # Transpose, drop duplicate 'rows' (original columns), and transpose back
df = df.sort_values(by='startTimestamp_y', ascending=True)

In [7]:
df

,gameId_home_p1_home_x,startTimestamp_home_p1_home_x,tournamentId_home_p1_home_x,round_home_p1_home_x,hasPlayerStatistics_home_p1_home_x,homeTeamId_home_p1_home_x,homeTeamFoundationDateTimestamp_home_p1_home_x,homeScoreNormalTime_home_p1_home_x,homeScorePeriod1_home_p1_home_x,homeScorePeriod2_home_p1_home_x,...,homeTotaltackle_y,awayTotaltackle_y,homeFreekicks_y,awayFreekicks_y,homeYellowcards_y,awayYellowcards_y,homeRedcards_y,awayRedcards_y,homeExpectedgoals_y,awayExpectedgoals_y
0,6698477,1442152800,83,25,True,1969,-2189808000.0,3,3.0,0.0,...,18.0,13.0,13.0,13.0,3.0,4.0,0.0,0.0,0.0,0.0
1,6698479,1442170800,83,25,True,1974,-2252016000.0,2,1.0,1.0,...,0.0,0.0,9.0,11.0,2.0,2.0,0.0,0.0,0.0,0.0
2,6698493,1442442600,83,26,True,1961,-2128550400.0,1,1.0,0.0,...,0.0,0.0,14.0,24.0,4.0,3.0,0.0,0.0,0.0,0.0
3,6698469,1442170800,83,25,True,5926,-2092176000.0,1,0.0,1.0,...,0.0,0.0,9.0,13.0,1.0,3.0,0.0,0.0,0.0,0.0
4,6698494,1442451600,83,26,True,1968,-1821398400.0,4,1.0,3.0,...,0.0,0.0,15.0,7.0,1.0,4.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5341,15235544,1777851000,83,14,True,21982,-1393113600.0,2,2.0,0.0,...,4.0,7.0,11.0,6.0,0.0,3.0,0.0,0.0,2.06,0.36
5342,15832733,1775685600,2929,1,True,3165,-390700800.0,1,0.0,1.0,...,11.0,12.0,9.0,13.0,1.0,3.0,0.0,0.0,1.54,0.83
7045,15832023,1775694600,91443,1,True,6968,-747014400.0,1,0.0,1.0,...,18.0,30.0,11.0,10.0,3.0,1.0,1.0,0.0,0.12,3.15
7046,15832084,1777586400,91443,3,True,6229,856915200.0,2,2.0,0.0,...,11.0,17.0,14.0,6.0,3.0,6.0,1.0,1.0,0.98,2.09


In [8]:
train_size = int(len(df) * 0.8)
train_data = df.iloc[:train_size]
test_data = df.iloc[train_size:]

X_train = train_data.filter(regex='_x$')
X_test = test_data.filter(regex='_x$')

print(X_train.columns)

#y_train = train_data['homePasses_y'] + train_data['awayPasses_y']
#y_test = test_data['homePasses_y'] + test_data['awayPasses_y']
#y_train = train_data['homeCornerkicks_y'] + train_data['awayCornerkicks_y']
#y_test = test_data['homeCornerkicks_y'] + test_data['awayCornerkicks_y']
y_train = (train_data['homeScoreNormalTime_y'] - train_data['awayScoreNormalTime_y'])
y_test = (test_data['homeScoreNormalTime_y'] - test_data['awayScoreNormalTime_y'])
scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Index(['gameId_home_p1_home_x', 'startTimestamp_home_p1_home_x',
       'tournamentId_home_p1_home_x', 'round_home_p1_home_x',
       'hasPlayerStatistics_home_p1_home_x', 'homeTeamId_home_p1_home_x',
       'homeTeamFoundationDateTimestamp_home_p1_home_x',
       'homeScoreNormalTime_home_p1_home_x', 'homeScorePeriod1_home_p1_home_x',
       'homeScorePeriod2_home_p1_home_x',
       ...
       'homeTotaltackle_away_p3_away_x', 'awayTotaltackle_away_p3_away_x',
       'homeFreekicks_away_p3_away_x', 'awayFreekicks_away_p3_away_x',
       'homeYellowcards_away_p3_away_x', 'awayYellowcards_away_p3_away_x',
       'homeRedcards_away_p3_away_x', 'awayRedcards_away_p3_away_x',
       'homeExpectedgoals_away_p3_away_x', 'awayExpectedgoals_away_p3_away_x'],
      dtype='str', length=449)


In [ ]:
model = RandomForestRegressor(n_estimators=100000, max_features='sqrt', random_state=42, verbose=1, n_jobs=8)
model.fit(X_train, y_train)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


building tree 1 of 100000building tree 2 of 100000
building tree 3 of 100000
building tree 4 of 100000

building tree 5 of 100000
building tree 6 of 100000
building tree 7 of 100000
building tree 8 of 100000
building tree 9 of 100000
building tree 10 of 100000
building tree 11 of 100000
building tree 12 of 100000
building tree 13 of 100000
building tree 14 of 100000
building tree 15 of 100000
building tree 16 of 100000
building tree 17 of 100000
building tree 18 of 100000
building tree 19 of 100000
building tree 20 of 100000
building tree 21 of 100000
building tree 22 of 100000
building tree 23 of 100000
building tree 24 of 100000


[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:    0.5s


building tree 25 of 100000
building tree 26 of 100000
building tree 27 of 100000
building tree 28 of 100000
building tree 29 of 100000
building tree 30 of 100000
building tree 31 of 100000
building tree 32 of 100000
building tree 33 of 100000
building tree 34 of 100000
building tree 35 of 100000
building tree 36 of 100000
building tree 37 of 100000
building tree 38 of 100000
building tree 39 of 100000
building tree 40 of 100000
building tree 41 of 100000
building tree 42 of 100000
building tree 43 of 100000
building tree 44 of 100000
building tree 45 of 100000
building tree 46 of 100000
building tree 47 of 100000
building tree 48 of 100000
building tree 49 of 100000
building tree 50 of 100000
building tree 51 of 100000
building tree 52 of 100000
building tree 53 of 100000
building tree 54 of 100000
building tree 55 of 100000
building tree 56 of 100000
building tree 57 of 100000
building tree 58 of 100000
building tree 59 of 100000
building tree 60 of 100000
building tree 61 of 100000
b

[Parallel(n_jobs=8)]: Done 112 tasks      | elapsed:    2.4s


...

[Parallel(n_jobs=8)]: Done 2032 tasks      | elapsed:   33.3s


building tree 2054 of 100000building tree 2055 of 100000

building tree 2056 of 100000
building tree 2057 of 100000
building tree 2058 of 100000
building tree 2059 of 100000
building tree 2060 of 100000
building tree 2061 of 100000
building tree 2062 of 100000
building tree 2063 of 100000
building tree 2064 of 100000
building tree 2065 of 100000
building tree 2066 of 100000
building tree 2067 of 100000
building tree 2068 of 100000
building tree 2069 of 100000
building tree 2070 of 100000
building tree 2071 of 100000
building tree 2072 of 100000
building tree 2073 of 100000
building tree 2074 of 100000
building tree 2075 of 100000
building tree 2076 of 100000
building tree 2077 of 100000
building tree 2078 of 100000
building tree 2079 of 100000
building tree 2080 of 100000
building tree 2081 of 100000
building tree 2082 of 100000
building tree 2083 of 100000
building tree 2084 of 100000
building tree 2085 of 100000
building tree 2086 of 100000
building tree 2087 of 100000
building tree 

[Parallel(n_jobs=8)]: Done 2576 tasks      | elapsed:   40.2s


building tree 2585 of 100000building tree 2586 of 100000

building tree 2587 of 100000
building tree 2588 of 100000
building tree 2589 of 100000
building tree 2590 of 100000
building tree 2591 of 100000
building tree 2592 of 100000
building tree 2593 of 100000
building tree 2594 of 100000
building tree 2595 of 100000
building tree 2596 of 100000
building tree 2597 of 100000
building tree 2598 of 100000
building tree 2599 of 100000
building tree 2600 of 100000
building tree 2601 of 100000
building tree 2602 of 100000
building tree 2603 of 100000
building tree 2604 of 100000
building tree 2605 of 100000
building tree 2606 of 100000
building tree 2607 of 100000
building tree 2608 of 100000
building tree 2609 of 100000
building tree 2610 of 100000
building tree 2611 of 100000
building tree 2612 of 100000
building tree 2613 of 100000
building tree 2614 of 100000
building tree 2615 of 100000
building tree 2616 of 100000
building tree 2617 of 100000building tree 2618 of 100000

building tree 

KeyboardInterrupt: 

In [15]:
model = MLPRegressor(hidden_layer_sizes=(5000, 2500, 1250), activation='relu', solver='adam', alpha=0.001, early_stopping=True, tol=0.0001, max_iter=2000, verbose=1)
model.fit(X_train, y_train)

Iteration 1, loss = 12.71066541
Validation score: 0.052751
Iteration 2, loss = 1.13221731
Validation score: 0.067158
Iteration 3, loss = 1.04875839
Validation score: 0.072002
Iteration 4, loss = 0.93627966
Validation score: -0.058561
Iteration 5, loss = 0.80998308
Validation score: -0.082756


c:\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(5000, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",2000
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [10]:
# 5. Make predictions
y_pred = model.predict(X_test)

# 6. Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(mse, r2)

print(y_test, y_pred)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:    0.1s


2.4337548241134748 0.06247659783425241
6821     0
6822     0
2890     2
2739     0
2891     0
        ..
5341     2
5342     1
7045    -6
7046    -1
5343    -1
Length: 1410, dtype: object [ 0.688  0.182  0.655 ... -0.317  0.082  0.609]


[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 1000 out of 1000 | elapsed:    0.2s finished


In [11]:
# Assume 'model' is your trained model and 'X' is your training data
importances = model.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easy sorting
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Sort and print
ordered_imp = feature_imp_df.sort_values(by='Importance', ascending=False)
print(ordered_imp[:50])

                                            Feature  Importance
6    homeTeamFoundationDateTimestamp_home_p1_home_x    0.018082
229                       homeTeamId_home_p1_away_x    0.017192
5                         homeTeamId_home_p1_home_x    0.014239
230  homeTeamFoundationDateTimestamp_home_p1_away_x    0.009558
163                        refereeId_away_p2_home_x    0.007700
139                       homePasses_away_p1_home_x    0.007277
176                       homePasses_away_p2_home_x    0.006805
252                       homePasses_home_p1_away_x    0.006749
289                       homePasses_home_p2_away_x    0.006728
401                       awayPasses_away_p2_away_x    0.006721
437                       homePasses_away_p3_away_x    0.006213
65                        homePasses_home_p2_home_x    0.006107
290                       awayPasses_home_p2_away_x    0.006081
438                       awayPasses_away_p3_away_x    0.006024
364                       awayPasses_awa

In [20]:
rf = RandomForestRegressor()
random_grid = {
    'n_estimators': [1000, 2000, 3000],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}
rf_random = GridSearchCV(estimator=rf, param_grid=random_grid, verbose=3, n_jobs=8)
rf_random.fit(X_train, y_train)
print(rf_random.best_params_)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


KeyboardInterrupt: 